# Final Project
CODA-RMT-016 - Group 2

## Import Library 

In [1]:
#Import library pandas 
import pandas as pd 

## Read CSV

In [2]:
#Read csv file as pandas dataframe
df = pd.read_csv('Telco-Customer-Churn.csv', delimiter=',')

In [3]:
df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


Convert 'TotalCharges' to type float

In [4]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', 0)

df['TotalCharges'] = df['TotalCharges'].astype(float)

In [5]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

## Initial Calculation

In [6]:
#Population Churn Rate
population = len(df)
churn_sum = (df['Churn'] == 'Yes').sum()
churn_rate = churn_sum / population * 100

#Find Population Sample
mask = (df['tenure'] == 1)
sample_promo = mask.sum()

#Find Population Churn Sample
sample_churn = (mask & (df['Churn'] == 'Yes')).sum()

#Calculation to get improvement absolute
converted_user = 0.05 * sample_churn
churn_after_promo = churn_sum - converted_user
churn_rate_after = churn_after_promo / len(df) * 100
improvement_absolute = churn_rate - churn_rate_after

print('improvement_absolute: ', improvement_absolute)

improvement_absolute:  0.26977140423115387


In [7]:
print('Population: ', population)
print('Population that churned: ', churn_sum)
print('Population Churn Rate: ', churn_rate, '%')
print('=== Sampling based on Tenure = 1 ===')
print('Sample Population:  ', sample_promo)
print('Sample Population Churn: ', sample_churn)
print('Conversion Rate 5%')
print('==== After Promotion ===')
print('Converted user: ', converted_user)
print('Population churn after promotion: ', churn_after_promo)
print('Population churn rate after promo: ', churn_rate_after)
print('Improvement absolute: ', improvement_absolute, '%')

Population:  7043
Population that churned:  1869
Population Churn Rate:  26.536987079369588 %
=== Sampling based on Tenure = 1 ===
Sample Population:   613
Sample Population Churn:  380
Conversion Rate 5%
==== After Promotion ===
Converted user:  19.0
Population churn after promotion:  1850.0
Population churn rate after promo:  26.267215675138434
Improvement absolute:  0.26977140423115387 %


## Function for Calculation & Analyze Column

Cari jumlah sample populasi

In [8]:
sample = {'Pay Method' : df['PaymentMethod'] == 'Electronic check', 
          'Contract' : df['Contract'] == 'Month-to-month', 
          'Online Security' : df['OnlineSecurity'] == 'No', 
          'Tech Support' : df['TechSupport'] == 'No',
          'Total Charge 0-83':(df['TotalCharges']>= 0) & (df['TotalCharges']<=83.47),
          'Senior Citizen':df['SeniorCitizen'] == 1,
          'Tenure 0-2':((df['tenure']>=0) & (df['tenure'])<=2),
          'Tenure 2-6':((df['tenure']>=2) & (df['tenure'])<=6),
          'Online Backup': df['OnlineBackup'] == 'No',
          'Device Protection': df['DeviceProtection'] == 'No'
}

tabel = []

for name, condition in sample.items():
    tabel.append({'condition':name, 'count':condition.sum()})

tabel_pop = pd.DataFrame(tabel)
sorted = tabel_pop.sort_values(by='count', ascending=True)
print(sorted)

           condition  count
4  Total Charge 0-83    705
5     Senior Citizen   1142
0         Pay Method   2365
8      Online Backup   3088
9  Device Protection   3095
3       Tech Support   3473
2    Online Security   3498
1           Contract   3875
7         Tenure 2-6   7043
6         Tenure 0-2   7043


Cari kombinasi sample populasi 10 % - 15%

In [9]:
from itertools import combinations
lower_threshold = 0.10 * len(df)
upper_threshold = 0.15 * len(df)
results = []

# coba kombinasi 2 sampai 4 kondisi
for r in range(1, 4):
    for combo in combinations(sample.items(), r):
        
        names = [name for name, cond in combo]
        masks = [cond for name, cond in combo]
        
        # gabung semua kondisi (AND)
        combined_mask = masks[0]
        for m in masks[1:]:
            combined_mask = combined_mask & m
        
        count = combined_mask.sum()
        all_names = ' & '.join(names)
        
        if lower_threshold <= count <= upper_threshold:
            
            results.append({
                'Combination': all_names,
                'Population': count,
                'Mask': combined_mask
            })

tabel_kombinasi = pd.DataFrame(results).sort_values(by='Population')
print(tabel_kombinasi)

                                       Combination  Population  \
0                                Total Charge 0-83         705   
4                   Total Charge 0-83 & Tenure 0-2         705   
5                   Total Charge 0-83 & Tenure 2-6         705   
13     Total Charge 0-83 & Tenure 0-2 & Tenure 2-6         705   
7           Contract & Senior Citizen & Tenure 0-2         807   
1                        Contract & Senior Citizen         807   
8           Contract & Senior Citizen & Tenure 2-6         807   
2                 Online Security & Senior Citizen         808   
10   Online Security & Senior Citizen & Tenure 2-6         808   
9    Online Security & Senior Citizen & Tenure 0-2         808   
11      Tech Support & Senior Citizen & Tenure 0-2         830   
3                    Tech Support & Senior Citizen         830   
12      Tech Support & Senior Citizen & Tenure 2-6         830   
6   Pay Method & Online Backup & Device Protection        1000   

         

Define variable kombinasi & cek jumlah populasi untuk tiap kombinasi

Function untuk hitung improvement rate

In [10]:
def calculate_improvement(df_sample, df_pop):
    #Population Churn Rate
    population = len(df_pop)
    churn_sum = (df_pop['Churn'] == 'Yes').sum()
    churn_rate = churn_sum / population * 100

    sample_churn = (df_sample['Churn'] == 'Yes').sum()

    #Calculation to get improvement absolute
    converted_user = 0.1 * sample_churn
    churn_after_promo = churn_sum - converted_user
    churn_rate_after = churn_after_promo / len(df_pop) * 100
    improvement_absolute = churn_rate - churn_rate_after
    #print('improvement:',improvement_absolute)
    return improvement_absolute, churn_rate_after

Hitung improvement rate untuk tiap kombinasi

In [11]:
all_improvement = []


for row in results:
    hasil, churn_rate_after = calculate_improvement(df[row['Mask']], df)
    all_improvement.append({'Combination Name': row['Combination'],
                            'Improvement absolute':hasil,
                            'Churn rate after promo': churn_rate_after})

tabel_improvement = pd.DataFrame(all_improvement).sort_values(by='Improvement absolute', ascending=False)
print(tabel_improvement)


                                  Combination Name  Improvement absolute  \
6   Pay Method & Online Backup & Device Protection              0.820673   
1                        Contract & Senior Citizen              0.626154   
7           Contract & Senior Citizen & Tenure 0-2              0.626154   
8           Contract & Senior Citizen & Tenure 2-6              0.626154   
11      Tech Support & Senior Citizen & Tenure 0-2              0.596337   
3                    Tech Support & Senior Citizen              0.596337   
12      Tech Support & Senior Citizen & Tenure 2-6              0.596337   
9    Online Security & Senior Citizen & Tenure 0-2              0.577879   
10   Online Security & Senior Citizen & Tenure 2-6              0.577879   
2                 Online Security & Senior Citizen              0.577879   
5                   Total Charge 0-83 & Tenure 2-6              0.518245   
4                   Total Charge 0-83 & Tenure 0-2              0.518245   
0           

In [37]:
#Menghitung jumlah pelanggan yang dikonversi
x = round(0.008* len(df))

#Menghitung rata-rata tagihan per pelanggan dalam sebulan
mean_c = df['MonthlyCharges'].mean().round()

#Menghitung profit retensi pelanggan
retensi = x * mean_c

#Batas harga promo
batas = retensi / 1000

print(f'Jumlah pelanggan kelompok segmentasi dengan improvement 0.8%: {x} orang')
print(f'Rata-rata tagihan pelanggan per bulan: ${mean_c}')
print(f'Profit hasil retensi: {retensi}')
print(f'Promosi yang diberikan per pelanggan: ${batas}')

Jumlah pelanggan kelompok segmentasi dengan improvement 0.8%: 56 orang
Rata-rata tagihan pelanggan per bulan: $65.0
Profit hasil retensi: 3640.0
Promosi yang diberikan per pelanggan: $3.64
